# The first DNN — `md_nn` and its dataset pipeline

The earliest detector build, from the `a_malware` workspace. It uses a separate `malware_detection` codebase (cloned in Step 1) and a smaller **1,368-feature** representation — the ancestor of the 10,000-feature detector in `dnn_detector.ipynb`.

The notebook has two halves: training and evaluating the model (including an Integrated Gradients look at which features drive it), then the pipeline that built that codebase's Drebin dataset from raw samples.

*Consolidated from `md_nn.ipynb` (originally in `a_malware`). Empty and duplicated loading cells were removed; outputs cleared.*

## Step 1 — Setup: clone the codebase and load the prepared datasets

In [ ]:
import time
import os.path as path
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import os

torch.manual_seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
!git clone https://github.com/mostafa-ja/malware_detection.git

In [ ]:
import torch
import torch.utils.data as data

class MyDataset(torch.utils.data.Dataset):
    def __init__(self, samples):
        self.samples = [(torch.tensor(data, dtype=torch.float32), label) for data, label in samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]




In [ ]:
train_dataset = torch.load('/content/malware_detection/datasets/train_dataset.pt')
train_Loader = data.DataLoader(train_dataset, batch_size=128, shuffle=False)
validation_dataset = torch.load('/content/malware_detection/datasets/validation_dataset.pt')
validation_Loader = data.DataLoader(validation_dataset, batch_size=10000, shuffle=False)
test_dataset = torch.load('/content/malware_detection/datasets/test_dataset.pt')
test_Loader = data.DataLoader(test_dataset, batch_size=10000, shuffle=False)

In [ ]:
print('train_dataset size : ', len(train_dataset))
print('validation_dataset size : ', len(validation_dataset))
print('test_dataset size : ', len(test_dataset))

## Step 2 — Look at the samples

In [ ]:
sample,label = next(iter(train_dataset))
print(sample.shape)
print(label)
print(sample)
print(sum(sample))

In [ ]:
sample,label = next(iter(validation_Loader))
print(sample.shape)


## Step 3 — The model

In [ ]:



class MalwareDetectionDNN(nn.Module):
    def __init__(self, input_size, n_classes, device='cpu', name='DNN', dense_hidden_units=[200, 200], dropout=0.6, smooth=False):
        """
        Construct malware detector

        Parameters
        ----------
        @param input_size: Integer, the dimentionality number of input vector
        @param n_classes: Integer, the number of classes, n=2
        @param device: String, 'cpu' or 'cuda'
        @param name: String, model name
        """
        super(MalwareDetectionDNN, self).__init__()
        self.input_size = input_size
        self.n_classes = n_classes
        self.name = name

        self.dropout = dropout
        self.smooth = smooth
        self.dense_hidden_units = dense_hidden_units

        self.dense_layers = []
        if len(self.dense_hidden_units) >= 1:
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        else:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                               self.dense_hidden_units[i + 1]))
        self.dense_layers.append(nn.Linear(self.dense_hidden_units[-1], self.n_classes))

        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('nn_model_layer_{}'.format(idx_i), dense_layer)

        if self.smooth:
            self.activation_func = F.selu  # partial(F.elu, alpha=self.alpha_)
        else:
            self.activation_func = F.relu

        self.model_save_path = path.join('/content/malware_detection/save/drebin/md_dnn'+ '_' + self.name,'model.pth')

    def forward(self, x):
        """
        Go through the neural network

        Parameters
        ----------
        @param x: 2D tensor, feature representation
        """
        for dense_layer in self.dense_layers[:-1]:
            x = self.activation_func(dense_layer(x))

        latent_representation = F.dropout(x, self.dropout, training=self.training)
        logits = self.dense_layers[-1](latent_representation)
        return logits




In [ ]:
model = MalwareDetectionDNN(1368,2)
model = model.to(device)
model

In [ ]:
from torchsummary import summary

summary(model, input_size=(1368,))


## Step 4 — Sanity checks on untrained outputs

In [ ]:
logits.shape

In [ ]:
y_train

In [ ]:
logits.argmax(1) == y_train

## Step 5 — Train

In [ ]:
lr =0.005
weight_decay = 0.
epochs = 50
dropout=0.6

optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
loss_fn = nn.CrossEntropyLoss()
best_avg_acc = 0.
best_epoch = 0
total_time = 0.
nbatches = len(train_Loader)
for i in range(epochs):
    model.train()
    losses, accuracies = [], []
    for idx_batch, (x_train, y_train) in enumerate(train_Loader):
        x_train = x_train.to(device)
        y_train = y_train.long().to(device)
        start_time = time.time()
        optimizer.zero_grad()
        logits = model.forward(x_train)
        loss_train = loss_fn(logits, y_train)
        loss_train.backward()
        optimizer.step()
        total_time = total_time + time.time() - start_time
        acc_train = (logits.argmax(1) == y_train).sum().item()
        acc_train /= x_train.size()[0]
        mins, secs = int(total_time / 60), int(total_time % 60)
        losses.append(loss_train.item())
        accuracies.append(acc_train)

        if True:
            print(f'Mini batch: {i * nbatches + idx_batch + 1}/{epochs * nbatches} | training time in {mins:.0f} minutes, {secs} seconds.')
            print(f'Training loss (batch level): {losses[-1]:.4f} | Train accuracy: {acc_train * 100:.2f}')

    model.eval()
    avg_acc_val = []
    with torch.no_grad():
        for x_val, y_val in validation_Loader:
            x_val = x_val.to(device)
            y_val = y_val.long().to(device)
            logits = model.forward(x_val)
            acc_val = (logits.argmax(1) == y_val).sum().item()
            acc_val /= x_val.size()[0]
            avg_acc_val.append(acc_val)
        avg_acc_val = np.mean(avg_acc_val)

    if avg_acc_val >= best_avg_acc:
        best_avg_acc = avg_acc_val
        best_epoch = i
        torch.save(model.state_dict(), 'model.pth')

    if True:
        print(f'Training loss (epoch level): {np.mean(losses):.4f} | Train accuracy: {np.mean(accuracies) * 100:.2f}')
        print(f'Validation accuracy: {avg_acc_val * 100:.2f} | The best validation accuracy: {best_avg_acc * 100:.2f} at epoch: {best_epoch}')

In [ ]:
test_sample,test_label = next(iter(test_Loader))
test_sample,test_label = test_sample.to(device),test_label.to(device)

model.load_state_dict(torch.load('/content/model.pth'))

model.eval()
with torch.no_grad():
    for x, y in test_Loader:
        x = x.to(device)
        logits = model.forward(x)
        confidences = F.softmax(logits, dim=-1)


y_pred = confidences.argmax(1).cpu().numpy()
y_true = y.cpu().numpy()

## Step 6 — Metrics

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
accuracy = accuracy_score(y_true, y_pred)
b_accuracy = balanced_accuracy_score(y_true, y_pred)
print("The accuracy on the test dataset is {:.5f}%".format(accuracy * 100))
print("The balanced accuracy on the test dataset is {:.5f}%".format(b_accuracy * 100))

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
fpr = fp / float(tn + fp)
fnr = fn / float(tp + fn)
f1 = f1_score(y_true, y_pred, average='binary')

print("Other evaluation metrics we may need:")
print("False Negative Rate (FNR) is {:.5f}%, False Positive Rate (FPR) is {:.5f}%, F1 score is {:.5f}%".format(fnr * 100, fpr * 100, f1 * 100))


## Step 7 — Which features drive it? Integrated Gradients (Captum)

In [ ]:
from captum.attr import IntegratedGradients

def get_important_attributes(self, test_data_producer, target_label=1):
        """
        get important attributes by using integrated gradients
        """
        attributions = []
        gt_labels = []

        def _ig_wrapper(_x):
            logits = self.forward(_x)
            return F.softmax(logits, dim=-1)
        ig = IntegratedGradients(_ig_wrapper)

        for i, (x, y) in enumerate(test_data_producer):
            x, y = utils.to_device(x.double(), y.long(), self.device)
            x.requires_grad = True
            baseline = torch.zeros_like(x, dtype=torch.double, device=self.device)
            attribution_bs = ig.attribute(x,
                                          baselines=baseline,
                                          target=target_label)
            attribution = torch.hstack(attribution_bs)
            attributions.append(attribution.clone().detach().cpu().numpy())
            gt_labels.append(y.clone().detach().cpu().numpy())
            np.save('./labels', np.concatenate(gt_labels))
        return np.vstack(attributions)

## Part 2 — Building the Drebin dataset for this codebase

The original notebook's own checkpoint note is kept:

# Test our result with the original code

In [ ]:
!rm -r /content/malware_detection/dataset/drebin

In [ ]:
%cd /content/malware_detection

In [ ]:
!pip install androguard
!pip install captum

import androguard
import captum

In [ ]:
import os
folder_path = '/content/malware_detection/datasets/naive_data'
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files in naive_data : {file_count}")

In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame
df_benign = pd.read_csv('/content/malware_detection/datasets/benign_sha256.csv')

# Extract the first column as a list
benign_sha256 = df_benign.iloc[:, 0].tolist()[:350]

In [ ]:
# download benign samples

import os
import requests
import concurrent.futures

# Replace with your actual API key in androidzoo
APIKEY = "717edc01a24bab5afe87835503feb736540c51352fe02bf2e487146caf0fa192"

# List of SHA256 hashes
sha256_list = benign_sha256

# Base URL for the download
base_url = "https://androzoo.uni.lu/api/download"

# Destination directory for downloaded APKs
download_dir = "/content/datasets/benign/"

# Create the download directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

# Function to download an APK file
def download_apk(sha256):
    params = {"apikey": APIKEY, "sha256": sha256}
    response = requests.get(base_url, params=params)
    if response.status_code == 200:
        filename = sha256
        file_path = os.path.join(download_dir, filename)
        with open(file_path, "wb") as apk_file:
            apk_file.write(response.content)
        print(f"Downloaded: {filename}")
    else:
        print(f"Failed to download APK with SHA256: {sha256}")

# Set the maximum number of concurrent downloads (adjust as needed)
max_concurrent_downloads = 40

# Use ThreadPoolExecutor to run download_apk concurrently
with concurrent.futures.ThreadPoolExecutor(max_concurrent_downloads) as executor:
    executor.map(download_apk, sha256_list)

print("Download process completed.")


In [ ]:
# Read the CSV file into a DataFrame
df_malicious = pd.read_csv('/content/malware_detection/datasets/malicious_sha256.csv')

# Extract the first column as a list
malicious_sha256 = df_malicious.iloc[:, 0].tolist()[:350]

In [ ]:
## download malicious samples

# List of SHA256 hashes
sha256_list = malicious_sha256

# Base URL for the download
base_url = "https://androzoo.uni.lu/api/download"

# Destination directory for downloaded APKs
download_dir = "/content/datasets/malicious/"

# Create the download directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

# Function to download an APK file
def download_apk(sha256):
    params = {"apikey": APIKEY, "sha256": sha256}
    response = requests.get(base_url, params=params)
    if response.status_code == 200:
        filename = sha256
        file_path = os.path.join(download_dir, filename)
        with open(file_path, "wb") as apk_file:
            apk_file.write(response.content)
        print(f"Downloaded: {filename}")
    else:
        print(f"Failed to download APK with SHA256: {sha256}")

# Set the maximum number of concurrent downloads (adjust as needed)
max_concurrent_downloads = 40

# Use ThreadPoolExecutor to run download_apk concurrently
with concurrent.futures.ThreadPoolExecutor(max_concurrent_downloads) as executor:
    executor.map(download_apk, sha256_list)

print("Download process completed.")

In [ ]:
!mv /content/datasets/benign/* /content/malware_detection/datasets/drebin/benign_samples/
!mv /content/datasets/malicious/* /content/malware_detection/datasets/drebin/malicious_samples/

In [ ]:
import os
folder_path = '/content/malware_detection/datasets/drebin/benign_samples'
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files in benign_samples : {file_count}")

In [ ]:
import os
folder_path = '/content/malware_detection/datasets/drebin/malicious_samples'
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files in malicious_samples : {file_count}")

In [ ]:
!ls

In [ ]:
import pickle

# Open and load the Pickle file
with open('/content/malware_detection/datasets/naive_data/629e1bc6a81d8b717dbcc0a942e60036ea61baf134e013b58bfdd2b2c06eeac7.feat', 'rb') as file:
    loaded_data = pickle.load(file)

# Access and use the loaded data
print(len(loaded_data))  # Example: Print the loaded data

In [ ]:
import sys

# Add the directory to the Python path
sys.path.append('/content/malware_detection/core/defense')

# Import the module
import dataset

In [ ]:
data = Dataset()
data.vocab_size

In [ ]:
import pickle

# Open the pickled file in binary read mode ('rb')
with open('/content/malware_detection/dataset/drebin/data.vocab', 'rb') as file:
    # Load the object from the file
    loaded_object = pickle.load(file)

# Now, 'loaded_object' contains the deserialized data from the pickled file
len(loaded_object)

In [ ]:
from core.defense import Dataset

# Dataset is a class
data = Dataset()
data.vocab_size

## Step 8 — Run the codebase's own test for this model

In [ ]:
!python -m examples.md_nn_test --cuda --cache --seed 0 --batch_size 128 --proc_number 10 --epochs 50 --max_vocab_size 10000 --dense_hidden_units "200,200" --weight_decay 0.0 --lr 0.001 --dropout 0.6